In [1]:
# voice_nav_qr.py
import os
import time
import cv2
import networkx as nx
import qrcode
from pyzbar import pyzbar
import pyttsx3
import winsound
import threading
import speech_recognition as sr

# ---------------- Text-to-Speech ----------------
class TextToSpeech:
    def __init__(self, tmp_dir="/tmp"):
        self.tmp_dir = tmp_dir
        os.makedirs(self.tmp_dir, exist_ok=True)

    def speak(self, text, lang='en', block=True):
        if not text:
            return
        try:
            engine = pyttsx3.init()
            engine.say(text)
            engine.runAndWait()
        except Exception as e:
            print("⚠ TTS error:", e)

def speak_realtime(text):
    def _speak():
        e = pyttsx3.init()
        e.say(text)
        e.runAndWait()
    threading.Thread(target=_speak, daemon=True).start()

def beep(freq, dur):
    try:
        threading.Thread(target=lambda: winsound.Beep(freq, dur), daemon=True).start()
    except Exception:
        pass

def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    sharpened = cv2.addWeighted(gray, 1.5, blurred, -0.5, 0)
    return sharpened

def draw_ui(frame, qr_found, qr_text=None):
    h, w, _ = frame.shape
    cv2.rectangle(frame, (0, 0), (w, 60), (0, 0, 0), -1)
    cv2.putText(frame, "Press 'q' to exit.", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    if qr_found and qr_text:
        cv2.rectangle(frame, (0, h - 60), (w, h), (0, 0, 0), -1)
        cv2.putText(frame, f"QR Detected: {qr_text}", (20, h - 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

last_guidance_time = 0
def guide_user(frame, x, y, w, h, readable=False, tts=None):
    global last_guidance_time
    if time.time() - last_guidance_time < 2:
        return
    last_guidance_time = time.time()
    qr_center_x = x + w // 2
    qr_center_y = y + h // 2
    frame_center_x = frame.shape[1] // 2
    frame_center_y = frame.shape[0] // 2
    dx = qr_center_x - frame_center_x
    dy = qr_center_y - frame_center_y
    area = w * h
    min_area_to_decode = 8000
    max_area_to_decode = 60000
    threshold = 50
    if readable:
        if tts:
            tts.speak("QR code centered. Hold still.", block=True)
        else:
            speak_realtime("QR code centered. Hold still.")
        beep(1200, 200)
    else:
        if dx > threshold:
            speak_realtime("Move camera to the right.")
            beep(600, 150)
        elif dx < -threshold:
            speak_realtime("Move camera to the left.")
            beep(800, 150)
        if dy > threshold:
            speak_realtime("Move camera down.")
            beep(400, 150)
        elif dy < -threshold:
            speak_realtime("Move camera up.")
            beep(1000, 150)
        if area < min_area_to_decode:
            speak_realtime("QR code detected but too far. Please move closer.")
        elif area > max_area_to_decode:
            speak_realtime("QR code too close. Move back slightly.")

class IndoorMap:
    def __init__(self):
        self.locations = {
            "Start": (0, -1),
            "T-junction": (0, 0),
            "N010": (-1, 0),
            "N009": (-2, 0),
            "N008": (-3, 0),
            "N011": (1, 0),
            "N012": (2, 0),
        }
        self.graph = nx.Graph()
        self.graph.add_nodes_from(self.locations.keys())
        self.graph.add_edges_from([
            ("Start", "T-junction"),
            ("T-junction", "N010"), ("N010", "N009"), ("N009", "N008"),
            ("T-junction", "N011"), ("N011", "N012"),
        ])

    def find_route(self, start, end):
        try:
            return nx.shortest_path(self.graph, start, end)
        except Exception:
            return []

class QRCodeGenerator:
    def __init__(self, output_dir="qrcodes"):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)

    def generate(self, locations):
        for loc in locations:
            img = qrcode.make(loc)
            path = os.path.join(self.output_dir, f"{loc}.png")
            img.save(path)
        print(f"✅ QR Codes generated in '{self.output_dir}' folder.")

class NavigationSystem:
    def __init__(self):
        self.tts = TextToSpeech()
        self.map = IndoorMap()
        self.qr_gen = QRCodeGenerator()
        self.current_location = None
        self.destination = None
        self.route = []
        self.dest_list = sorted(self.map.locations.keys())
        self.left_rooms = {"N008", "N009", "N010"}
        self.right_rooms = {"N011", "N012"}
        self.recognizer = sr.Recognizer()

    def speak_destinations(self):
        lines = []
        for i, dest in enumerate(self.dest_list, start=1):
            lines.append(f"{i}. {dest}")
        lines.append(f"{len(self.dest_list) + 1}. Exit")  # ADD EXIT OPTION
        announce = "Available destinations are. " + ". ".join(lines) + ". Say the number of the destination."
        self.tts.speak(announce, block=True)

    def listen_for_choice(self, timeout=8, phrase_time_limit=6):
        mic = sr.Microphone()
        while True:
            with mic as source:
                print("🎤 Calibrating mic for ambient noise...")
                self.recognizer.adjust_for_ambient_noise(source, duration=1)
                print("🎤 Listening now...")
                speak_realtime("Listening now, please say a number.")
                try:
                    audio = self.recognizer.listen(source, timeout=timeout, phrase_time_limit=phrase_time_limit)
                except sr.WaitTimeoutError:
                    print("⚠ Listening timed out, retrying...")
                    continue
            try:
                text = self.recognizer.recognize_google(audio).lower().strip()
                print("✅ Recognized speech:", text)
            except sr.UnknownValueError:
                print("⚠ Could not understand audio, retrying...")
                continue
            except sr.RequestError:
                print("⚠ API unavailable or no internet")
                continue

            word2num = {
                'one': 1, '1': 1, 'two': 2, 'to': 2, 'too': 2, '2': 2,
                'three': 3, '3': 3, 'four': 4, 'for': 4, '4': 4,
                'five': 5, '5': 5, 'six': 6, '6': 6, 'seven': 7, '7': 7,
                'eight': 8, '8': 8, 'nine': 9, '9': 9, 'ten': 10, '10': 10
            }
            for token in text.split():
                if token in word2num:
                    return word2num[token]
            for chunk in text.split():
                try:
                    return int(''.join(ch for ch in chunk if ch.isdigit()))
                except:
                    continue

    def set_destination_by_index(self, index):
        try:
            idx = int(index)
        except:
            return False

    # Handle exit option
        if idx == 8:
            self.tts.speak("Exiting navigation system. Goodbye.")
            exit(0)

        if idx < 1 or idx > len(self.dest_list):
            return False

        dest = self.dest_list[idx - 1]
        self.destination = dest
        start = self.current_location if self.current_location else "Start"
        self.route = self.map.find_route(start, self.destination)
        self.tts.speak(f"Destination set to {dest}. Opening camera now.")
        return True


    def process_location(self, code):
        if code not in self.map.locations:
            self.tts.speak("Scanned code not recognized. Please try again.")
            return "invalid"

        self.current_location = code
        self.tts.speak(f"You are at {code}.")

        if not self.destination:
            return "invalid"

        if self.current_location == self.destination:
            self.tts.speak(f"You have arrived at {self.destination}.")
            self.destination = None
            self.route = []
            return "arrived"

        self.route = self.map.find_route(self.current_location, self.destination)
        if not self.route or len(self.route) < 2:
            self.tts.speak("No route found. Please try another destination.")
            return "invalid"

        try:
            next_step_index = self.route.index(self.current_location) + 1
        except ValueError:
            self.tts.speak(f"Proceed towards {self.destination}.")
            return "continue"

        if next_step_index < len(self.route):
            next_step = self.route[next_step_index]
            if self.current_location == "T-junction":
                if next_step in self.left_rooms:
                    self.tts.speak("From T-junction, turn left and walk straight to " + next_step)
                elif next_step in self.right_rooms:
                    self.tts.speak("From T-junction, turn right and walk straight to " + next_step)
                else:
                    self.tts.speak(f"From T-junction, walk straight to {next_step}")
            else:
                self.tts.speak(f"From {self.current_location}, walk straight to {next_step}.")
        else:
            self.tts.speak(f"Proceed straight until you reach {self.destination}.")
        return "continue"

    def scan_with_camera(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.tts.speak("Camera not found.")
            return
        speak_realtime("Scanning mode started. Show QR code. Press Q to quit.")
        qr_text = None
        stop_reason = None
        last_processed_time = 0.0
        min_process_gap = 0.6
        while True:
            ret, frame = cap.read()
            if not ret:
                time.sleep(0.02)
                continue
            processed = preprocess_frame(frame)
            barcodes = pyzbar.decode(processed)
            detected = False
            for barcode in barcodes:
                x, y, w, h = barcode.rect
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 4)
                area = w * h
                min_area_to_decode = 8000
                try:
                    qr_data = barcode.data.decode("utf-8")
                except:
                    qr_data = None
                if qr_data:
                    detected = True
                    if area >= min_area_to_decode:
                        now = time.time()
                        if now - last_processed_time >= min_process_gap:
                            last_processed_time = now
                            qr_text = qr_data
                            guide_user(frame, x, y, w, h, readable=True, tts=self.tts)
                            print("📷 Scanned QR Code:", qr_text)
                            status = self.process_location(qr_text)
                            if status == "arrived":
                                stop_reason = "arrived"
                                break
                    else:
                        guide_user(frame, x, y, w, h, readable=False, tts=self.tts)
            draw_ui(frame, detected, qr_text)
            cv2.imshow("QR Scanner (press 'q' to quit)", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                stop_reason = "quit"
                break
            if stop_reason:
                break
        cap.release()
        cv2.destroyAllWindows()
        if stop_reason == "arrived":
            self.tts.speak("Camera closed. You may now select a new destination.")
        else:
            self.tts.speak("Camera closed. You may speak a new destination.")

    def run_voice_loop(self):
        self.tts.speak("Welcome to the QR indoor navigation system.")
        while True:
            time.sleep(0.5)
            self.speak_destinations()
            choice = self.listen_for_choice(timeout=8, phrase_time_limit=6)

            if isinstance(choice, int):
                if choice == 8:  # Exit option
                    self.tts.speak("Exiting navigation system. Goodbye.")
                    break  # stop the loop cleanly

                ok = self.set_destination_by_index(choice)
                if not ok:
                    self.tts.speak("Invalid selection. Please try again.")
                    continue
                self.scan_with_camera()
            else:
                self.tts.speak("Sorry, I couldn't understand. Please try again.")


if __name__ == "__main__":
    nav = NavigationSystem()
    nav.run_voice_loop()


🎤 Calibrating mic for ambient noise...
🎤 Listening now...
⚠ Could not understand audio, retrying...
🎤 Calibrating mic for ambient noise...
🎤 Listening now...
✅ Recognized speech: one
🎤 Calibrating mic for ambient noise...
🎤 Listening now...
✅ Recognized speech: 8
